# 🛒 Reglas de Asociación — Notebook Completo
> **Curso:** Machine Learning / Business Intelligence  
> **Objetivo:** Descubrir patrones de co-ocurrencia en transacciones usando Apriori y FP-Growth

---

## ¿Qué aprenderás?
1. Explorar y preprocesar datos transaccionales
2. Codificar transacciones como matriz binaria
3. Minar ítems frecuentes con **Apriori** y **FP-Growth**
4. Calcular e interpretar **Soporte**, **Confianza** y **Lift**
5. Visualizar reglas como red de asociaciones
6. Traducir reglas en decisiones de negocio

---
### Contexto de negocio
Somos analistas de un supermercado online. Queremos encontrar qué productos se compran juntos para mejorar:
- 📦 **Layout** de la tienda y recomendaciones en pantalla
- 📧 **Email marketing** personalizado
- 🎁 **Bundles** y descuentos cruzados

## 1. Instalación y Configuración

In [ ]:
!pip install mlxtend networkx --quiet

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import networkx as nx
import warnings
warnings.filterwarnings('ignore')

from mlxtend.preprocessing import TransactionEncoder
from mlxtend.frequent_patterns import apriori, fpgrowth, association_rules

# ── Estilo de gráficos ────────────────────────────────────────────
plt.rcParams.update({
    'figure.facecolor': '#18181b',
    'axes.facecolor':   '#27272a',
    'axes.edgecolor':   '#3f3f46',
    'text.color':       '#e4e4e7',
    'axes.labelcolor':  '#a1a1aa',
    'xtick.color':      '#71717a',
    'ytick.color':      '#71717a',
    'grid.color':       '#3f3f46',
    'grid.alpha':        0.4,
    'axes.titlesize':    13,
    'axes.titlepad':     10,
    'figure.titlesize':  14,
})

EMERALD  = '#10b981'
AMBER    = '#f59e0b'
BLUE     = '#3b82f6'
PURPLE   = '#8b5cf6'
RED      = '#ef4444'
ZINC400  = '#a1a1aa'

np.random.seed(42)
print('✅ Librerías cargadas correctamente')

---
## 2. Dataset — Supermercado Online

Generamos **500 transacciones** con patrones realistas embedidos.

> 💡 **Truco didáctico:** insertamos patrones conocidos a propósito (ej. {pañales → cerveza}) para que los podamos encontrar y validar las métricas a mano.

In [ ]:
# Probabilidades base de cada ítem (frecuencia relativa en el supermercado)
ITEM_PROBS = {
    'leche':          0.62,
    'pan':            0.58,
    'huevos':         0.54,
    'mantequilla':    0.42,
    'queso':          0.38,
    'yogur':          0.33,
    'café':           0.44,
    'azúcar':         0.28,
    'arroz':          0.32,
    'pasta':          0.30,
    'pollo':          0.38,
    'carne':          0.28,
    'verduras':       0.48,
    'frutas':         0.52,
    'jugo':           0.32,
    'refresco':       0.27,
    'cerveza':        0.22,
    'vino':           0.18,
    'chips':          0.28,
    'chocolate':      0.32,
    'galletas':       0.24,
    'salsa_tomate':   0.25,
    'aceite':         0.30,
    'detergente':     0.22,
    'pañales':        0.12,
}

N = 500
transactions = []

# Transacciones base (probabilísticas)
for _ in range(N):
    basket = [item for item, p in ITEM_PROBS.items() if np.random.random() < p]
    if len(basket) >= 1:
        transactions.append(sorted(basket))

# Patrones embedidos — subir artificialmente la co-ocurrencia
patterns = [
    (['pañales', 'cerveza', 'chips'],          60),   # clásico
    (['leche', 'pan', 'mantequilla', 'huevos'], 55),  # desayuno
    (['pasta', 'salsa_tomate', 'queso'],        50),  # combo italiano
    (['pollo', 'arroz', 'verduras'],            45),  # almuerzo saludable
    (['café', 'azúcar', 'leche'],               48),  # café con leche
    (['vino', 'queso', 'galletas'],             30),  # aperitivo
    (['chocolate', 'galletas', 'leche'],        35),  # merienda
]
for items_list, reps in patterns:
    for _ in range(reps):
        extra = [i for i, p in ITEM_PROBS.items()
                 if np.random.random() < p * 0.4 and i not in items_list]
        transactions.append(sorted(set(items_list + extra)))

np.random.shuffle(transactions)

print(f'Total de transacciones: {len(transactions)}')
print(f'Ejemplo de transacción:  {transactions[0]}')
print(f'Ítem más largo:          {max(transactions, key=len)} ({max(len(t) for t in transactions)} ítems)')

---
## 3. Exploración de Datos (EDA)

In [ ]:
# ── Estadísticas generales ────────────────────────────────────────
all_items  = [item for t in transactions for item in t]
basket_len = [len(t) for t in transactions]

item_counts = pd.Series(all_items).value_counts()
item_support = item_counts / len(transactions)

print('═' * 50)
print(f'  Transacciones totales  : {len(transactions)}')
print(f'  Ítems únicos           : {item_counts.nunique()}')
print(f'  Promedio ítems/canasta : {np.mean(basket_len):.1f}')
print(f'  Densidad de la matriz  : {sum(basket_len) / (len(transactions) * item_counts.nunique()):.3f}')
print('═' * 50)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Exploración del Dataset — Supermercado Online', fontsize=14, color='#e4e4e7')

# Frecuencia de ítems
top_items = item_support.head(15)
colors_bar = [EMERALD if s > 0.4 else BLUE if s > 0.25 else ZINC400 for s in top_items]
axes[0].barh(top_items.index[::-1], top_items.values[::-1], color=colors_bar[::-1], edgecolor='none')
axes[0].set_xlabel('Soporte (fracción de transacciones)')
axes[0].set_title('Top 15 Ítems por Soporte')
axes[0].axvline(0.4, color=AMBER, linewidth=1, linestyle='--', label='umbral 0.40')
axes[0].legend()
axes[0].grid(axis='x')

# Distribución del tamaño de canasta
bins = range(1, max(basket_len) + 2)
axes[1].hist(basket_len, bins=bins, color=PURPLE, alpha=0.8, edgecolor='#18181b', rwidth=0.85)
axes[1].axvline(np.mean(basket_len), color=AMBER, linewidth=2, linestyle='--',
                label=f'media = {np.mean(basket_len):.1f}')
axes[1].set_xlabel('Número de ítems en la canasta')
axes[1].set_ylabel('Frecuencia')
axes[1].set_title('Distribución del Tamaño de Canasta')
axes[1].legend()
axes[1].grid(axis='y')

plt.tight_layout()
plt.show()

---
## 4. Preprocesamiento — Codificación Binaria

Los algoritmos de AR requieren una **matriz binaria** donde:
- Cada **fila** = una transacción
- Cada **columna** = un ítem posible
- Cada celda = `1` si el ítem está en la transacción, `0` si no

```
              leche  pan  huevos  mantequilla  queso  ...
Transacción 1   1     1     0         1          0
Transacción 2   1     0     1         0          1
Transacción 3   0     1     1         1          0
```

> ⚠️ **Diferencia clave con Sequence Mining:** aquí el **orden no importa**. {pan, leche} y {leche, pan} son la misma transacción.

In [ ]:
# Codificación con TransactionEncoder de mlxtend
te = TransactionEncoder()
te_array = te.fit_transform(transactions)
df_bin = pd.DataFrame(te_array, columns=te.columns_)

print(f'Forma de la matriz binaria: {df_bin.shape}')
print(f'Filas = transacciones, Columnas = ítems únicos')
print()
df_bin.head()

In [ ]:
# Visualizar la matriz binaria para las primeras 20 transacciones
fig, ax = plt.subplots(figsize=(16, 5))
subset = df_bin.iloc[:20].astype(int)
sns.heatmap(subset, cmap=['#27272a', EMERALD], linewidths=0.3, linecolor='#18181b',
            cbar=False, ax=ax, yticklabels=range(1, 21))
ax.set_title('Matriz Binaria — Primeras 20 Transacciones (verde = presente)', pad=12)
ax.set_xlabel('Ítems')
ax.set_ylabel('Transacción #')
plt.xticks(rotation=45, ha='right', fontsize=9)
plt.tight_layout()
plt.show()

---
## 5. Algoritmo Apriori — Ítems Frecuentes

### ¿Cómo funciona?
1. Encuentra todos los **1-ítems** con soporte ≥ `min_support`
2. Genera candidatos de tamaño `k+1` combinando los frecuentes de tamaño `k`
3. Poda cualquier candidato cuyo **subconjunto no sea frecuente** (principio anti-monótono)
4. Repite hasta que no haya candidatos

| min_support | Efecto |
|---|---|
| Alto (0.30+) | Pocos ítems, reglas muy comunes pero obvias |
| Medio (0.10–0.20) | Balance entre cobertura y novedad |
| Bajo (<0.05) | Muchos ítems, reglas de nicho, mayor costo computacional |

In [ ]:
MIN_SUPPORT = 0.07   # ← prueba con 0.05, 0.10, 0.15

frequent_apriori = apriori(df_bin, min_support=MIN_SUPPORT, use_colnames=True)
frequent_apriori['length'] = frequent_apriori['itemsets'].apply(len)
frequent_apriori = frequent_apriori.sort_values('support', ascending=False)

print(f'Ítems frecuentes encontrados con soporte ≥ {MIN_SUPPORT}: {len(frequent_apriori)}')
print()
print('Distribución por tamaño:')
print(frequent_apriori['length'].value_counts().sort_index().to_string())
print()
print('Top 10 ítems frecuentes:')
frequent_apriori.head(10)[['itemsets','support','length']].to_string(index=False)

In [ ]:
# Efecto del umbral de soporte en el número de ítems frecuentes
thresholds = np.arange(0.03, 0.35, 0.02)
counts = [len(apriori(df_bin, min_support=t, use_colnames=True)) for t in thresholds]

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(thresholds, counts, color=EMERALD, linewidth=2.5, marker='o', markersize=5)
ax.fill_between(thresholds, counts, alpha=0.15, color=EMERALD)
ax.axvline(MIN_SUPPORT, color=AMBER, linewidth=1.5, linestyle='--', label=f'umbral actual = {MIN_SUPPORT}')
ax.set_xlabel('min_support')
ax.set_ylabel('Número de ítems frecuentes')
ax.set_title('Sensibilidad al Umbral de Soporte (Apriori)')
ax.legend()
ax.grid(True)
plt.tight_layout()
plt.show()

---
## 6. Algoritmo FP-Growth — Comparación con Apriori

**FP-Growth** construye un árbol comprimido de prefijos (**FP-Tree**) y mina patrones sin generar candidatos explícitamente.

| Dimensión | Apriori | FP-Growth |
|---|---|---|
| Pasadas a la BD | Múltiples | 2 |
| Generación de candidatos | Sí | No |
| Uso de memoria | Bajo | Mayor (árbol) |
| Velocidad (datasets grandes) | Lento | Rápido |
| Resultado | Idéntico | Idéntico |

In [ ]:
import time

t0 = time.perf_counter()
frequent_apriori2 = apriori(df_bin, min_support=MIN_SUPPORT, use_colnames=True)
t_apriori = time.perf_counter() - t0

t0 = time.perf_counter()
frequent_fp = fpgrowth(df_bin, min_support=MIN_SUPPORT, use_colnames=True)
t_fp = time.perf_counter() - t0

print(f'Apriori  → {len(frequent_apriori2):4d} ítems frecuentes en {t_apriori*1000:.1f} ms')
print(f'FP-Growth → {len(frequent_fp):4d} ítems frecuentes en {t_fp*1000:.1f} ms')
print(f'Speedup FP-Growth: {t_apriori/t_fp:.1f}x')
print()
# Verificar que los resultados son idénticos
sets_a = set(frozenset(i) for i in frequent_apriori2['itemsets'])
sets_f = set(frozenset(i) for i in frequent_fp['itemsets'])
print(f'¿Resultados idénticos? {sets_a == sets_f}  ← siempre debe ser True')

---
## 7. Reglas de Asociación y Métricas

### Definición formal
Una regla `X → Y` significa: *"cuando ocurre X, tiende a ocurrir Y"*

| Métrica | Fórmula | Interpretación |
|---|---|---|
| **Soporte** | `P(X ∪ Y)` | ¿Qué tan común es la combinación en el total? |
| **Confianza** | `P(Y \| X) = P(X ∪ Y) / P(X)` | Dado X, ¿qué tan seguro es Y? |
| **Lift** | `conf(X→Y) / P(Y)` | ¿La regla es mejor que el azar? |
| **Leverage** | `P(X ∪ Y) - P(X)·P(Y)` | Exceso de co-ocurrencia vs independencia |
| **Conviction** | `(1 - P(Y)) / (1 - conf(X→Y))` | Qué tan segura es la implicación (∞ = perfecta) |

### Guía de Lift
- `lift = 1` → X e Y son **independientes** (la regla no agrega valor)
- `lift > 1` → **asociación positiva** (X favorece Y)
- `lift < 1` → **asociación negativa** (X suprime Y)

In [ ]:
MIN_CONFIDENCE = 0.40
MIN_LIFT       = 1.20

rules = association_rules(frequent_fp, metric='lift', min_threshold=MIN_LIFT)
rules = rules[rules['confidence'] >= MIN_CONFIDENCE].copy()
rules = rules.sort_values('lift', ascending=False)

# Formatear para lectura
rules['antecedents_str'] = rules['antecedents'].apply(lambda x: ', '.join(sorted(x)))
rules['consequents_str'] = rules['consequents'].apply(lambda x: ', '.join(sorted(x)))
rules['rule'] = rules['antecedents_str'] + '  →  ' + rules['consequents_str']

print(f'Reglas encontradas (conf ≥ {MIN_CONFIDENCE}, lift ≥ {MIN_LIFT}): {len(rules)}')
print()

display_cols = ['rule', 'support', 'confidence', 'lift', 'leverage']
rules[display_cols].head(15).style \
    .format({'support': '{:.3f}', 'confidence': '{:.3f}', 'lift': '{:.2f}', 'leverage': '{:.4f}'}) \
    .bar(subset=['lift'], color='#10b981', vmin=1) \
    .bar(subset=['confidence'], color='#3b82f6', vmin=0)

In [ ]:
# ── Cálculo manual para validar ───────────────────────────────────
def calc_rule_manual(transactions, antecedent, consequent):
    """Calcula soporte, confianza y lift a mano para verificar mlxtend."""
    n = len(transactions)
    ant_set = set(antecedent)
    con_set = set(consequent)
    both = sum(1 for t in transactions if ant_set <= set(t) and con_set <= set(t))
    ant_only = sum(1 for t in transactions if ant_set <= set(t))
    con_only = sum(1 for t in transactions if con_set <= set(t))
    support    = both / n
    confidence = both / ant_only if ant_only > 0 else 0
    lift       = confidence / (con_only / n) if con_only > 0 else 0
    return support, confidence, lift

# Verificar regla: {pañales} → {cerveza}
sup, conf, lift = calc_rule_manual(transactions, ['pañales'], ['cerveza'])
print('Regla: {pañales} → {cerveza}')
print(f'  Soporte    = {sup:.4f}  ({sup*100:.1f}% de transacciones contienen ambos)')
print(f'  Confianza  = {conf:.4f}  ({conf*100:.1f}% de compradores de pañales también compran cerveza)')
print(f'  Lift       = {lift:.4f}  ({lift:.1f}x más probable que por azar)')
print()
# Verificar regla: {pan, mantequilla} → {leche}
sup2, conf2, lift2 = calc_rule_manual(transactions, ['pan', 'mantequilla'], ['leche'])
print('Regla: {pan, mantequilla} → {leche}')
print(f'  Soporte    = {sup2:.4f}')
print(f'  Confianza  = {conf2:.4f}')
print(f'  Lift       = {lift2:.4f}')

In [ ]:
# ── Distribución de métricas ──────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
fig.suptitle('Distribución de Métricas en las Reglas Encontradas')

for ax, col, color, title in zip(
    axes,
    ['support', 'confidence', 'lift'],
    [EMERALD, BLUE, PURPLE],
    ['Soporte', 'Confianza', 'Lift']
):
    ax.hist(rules[col], bins=20, color=color, alpha=0.85, edgecolor='#18181b')
    ax.axvline(rules[col].mean(), color=AMBER, linewidth=1.5, linestyle='--',
               label=f'media={rules[col].mean():.2f}')
    if col == 'lift':
        ax.axvline(1.0, color=RED, linewidth=1, linestyle=':', label='lift=1 (azar)')
    ax.set_title(title)
    ax.set_xlabel(col)
    ax.legend(fontsize=8)
    ax.grid(axis='y')

plt.tight_layout()
plt.show()

---
## 8. Visualizaciones

In [ ]:
# ── Scatter: Soporte vs Confianza (tamaño = lift) ─────────────────
fig, ax = plt.subplots(figsize=(10, 6))

sc = ax.scatter(
    rules['support'],
    rules['confidence'],
    c=rules['lift'],
    s=rules['lift'] * 30,
    cmap='plasma',
    alpha=0.75,
    edgecolors='none'
)

cbar = plt.colorbar(sc, ax=ax)
cbar.set_label('Lift', color='#e4e4e7')
cbar.ax.yaxis.set_tick_params(color='#71717a')

# Anotar las top 5 reglas por lift
top5 = rules.nlargest(5, 'lift')
for _, row in top5.iterrows():
    ax.annotate(
        f"{row['antecedents_str']} → {row['consequents_str']}",
        (row['support'], row['confidence']),
        textcoords='offset points', xytext=(8, 4),
        fontsize=7, color='#fbbf24',
        bbox=dict(boxstyle='round,pad=0.2', fc='#18181b', alpha=0.7)
    )

ax.set_xlabel('Soporte')
ax.set_ylabel('Confianza')
ax.set_title('Soporte vs Confianza — color y tamaño = Lift')
ax.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
# ── Heatmap de Lift entre top ítems ──────────────────────────────
top_items_lift = list(item_support.head(12).index)
lift_matrix = pd.DataFrame(index=top_items_lift, columns=top_items_lift, dtype=float)

for a in top_items_lift:
    for b in top_items_lift:
        if a == b:
            lift_matrix.loc[a, b] = np.nan
            continue
        row = rules[(rules['antecedents_str'] == a) & (rules['consequents_str'] == b)]
        lift_matrix.loc[a, b] = row['lift'].values[0] if len(row) > 0 else 1.0

fig, ax = plt.subplots(figsize=(10, 8))
mask = lift_matrix.isna()
sns.heatmap(
    lift_matrix.astype(float), ax=ax,
    cmap='RdYlGn', center=1.0, vmin=0.5, vmax=2.5,
    linewidths=0.3, linecolor='#18181b',
    mask=mask, annot=True, fmt='.2f', annot_kws={'size': 8}
)
ax.set_title('Heatmap de Lift entre los 12 ítems más frecuentes\n(verde > 1 = asociación positiva, rojo < 1 = negativa)')
plt.tight_layout()
plt.show()

In [ ]:
# ── Red de Asociaciones ───────────────────────────────────────────
top_rules = rules.nlargest(30, 'lift')

G = nx.DiGraph()
for _, row in top_rules.iterrows():
    for a in row['antecedents']:
        for c in row['consequents']:
            G.add_edge(a, c, weight=row['lift'], confidence=row['confidence'])

fig, ax = plt.subplots(figsize=(13, 9))
ax.set_facecolor('#18181b')

pos = nx.spring_layout(G, seed=42, k=2.2)

# Nodos — tamaño = soporte del ítem
node_sizes = [item_support.get(n, 0.1) * 3500 for n in G.nodes()]
nx.draw_networkx_nodes(G, pos, node_color=BLUE, node_size=node_sizes, alpha=0.85, ax=ax)

# Aristas — grosor = lift
edges = G.edges(data=True)
edge_widths = [d['weight'] * 0.8 for _, _, d in edges]
edge_alphas = [min(d['confidence'], 1.0) for _, _, d in edges]
nx.draw_networkx_edges(G, pos, width=edge_widths, alpha=0.6,
                       edge_color=EMERALD, arrows=True,
                       arrowstyle='->', arrowsize=12, ax=ax)

nx.draw_networkx_labels(G, pos, font_size=9, font_color='#f4f4f5', ax=ax)

ax.set_title('Red de Reglas de Asociación\n(grosor del arco = Lift, tamaño del nodo = Soporte del ítem)', pad=14)
ax.axis('off')
plt.tight_layout()
plt.show()

---
## 9. Análisis de Negocios

Las métricas son solo números. El valor real está en **traducirlas en acciones**.

In [ ]:
# ── Top reglas por lift ───────────────────────────────────────────
print('🏆 TOP 10 REGLAS POR LIFT (asociaciones más sorprendentes)')
print('═' * 80)
for _, row in rules.nlargest(10, 'lift').iterrows():
    print(f"  {row['antecedents_str']:25s} →  {row['consequents_str']:20s}"
          f"  lift={row['lift']:.2f}  conf={row['confidence']:.0%}  sup={row['support']:.3f}")

In [ ]:
# ── Reglas útiles para un ítem específico ────────────────────────
TARGET_ITEM = 'leche'   # ← cambia este ítem

print(f'🎯 REGLAS DONDE "{TARGET_ITEM}" ES ANTECEDENTE (si compra leche → también compra...)')
print('═' * 75)
r_ant = rules[rules['antecedents_str'].str.contains(TARGET_ITEM, na=False)] \
              .sort_values('lift', ascending=False)
for _, row in r_ant.head(8).iterrows():
    print(f"  → {row['consequents_str']:25s}  lift={row['lift']:.2f}  conf={row['confidence']:.0%}")

print()
print(f'🎯 REGLAS DONDE "{TARGET_ITEM}" ES CONSECUENTE (compradores de X también compran leche)')
print('═' * 75)
r_con = rules[rules['consequents_str'].str.contains(TARGET_ITEM, na=False)] \
              .sort_values('lift', ascending=False)
for _, row in r_con.head(8).iterrows():
    print(f"  {row['antecedents_str']:25s} →  leche  lift={row['lift']:.2f}  conf={row['confidence']:.0%}")

In [ ]:
# ── Asimetría de las reglas ───────────────────────────────────────
# A→B y B→A tienen la misma confianza? No necesariamente
print('↔️  ASIMETRÍA: Comparación A→B vs B→A')
print('═' * 70)
symmetric_pairs = []
for _, r1 in rules.iterrows():
    # Buscar regla inversa
    inv = rules[
        (rules['antecedents_str'] == r1['consequents_str']) &
        (rules['consequents_str'] == r1['antecedents_str'])
    ]
    if len(inv) > 0 and r1['antecedents_str'] < r1['consequents_str']:
        r2 = inv.iloc[0]
        symmetric_pairs.append({
            'A → B': f"{r1['antecedents_str']} → {r1['consequents_str']}",
            'conf(A→B)': f"{r1['confidence']:.0%}",
            'conf(B→A)': f"{r2['confidence']:.0%}",
            'lift':       f"{r1['lift']:.2f}",
        })

if symmetric_pairs:
    df_sym = pd.DataFrame(symmetric_pairs)
    print(df_sym.to_string(index=False))
else:
    print('  No se encontraron pares simétricos con los umbrales actuales.')

In [ ]:
# ── Resumen ejecutivo de recomendaciones ─────────────────────────
print('📋 RESUMEN EJECUTIVO — RECOMENDACIONES DE NEGOCIO')
print('═' * 65)

top3_lift  = rules.nlargest(3, 'lift')
top3_conf  = rules.nlargest(3, 'confidence')
top3_sup   = rules.nlargest(3, 'support')

print('\n🥇 Para RECOMENDACIONES SORPRENDENTES (mayor lift):')
for _, r in top3_lift.iterrows():
    print(f"   {r['antecedents_str']} → {r['consequents_str']}  (lift={r['lift']:.2f})")

print('\n🔒 Para CROSS-SELL DE ALTA CONVERSIÓN (mayor confianza):')
for _, r in top3_conf.iterrows():
    print(f"   {r['antecedents_str']} → {r['consequents_str']}  (conf={r['confidence']:.0%})")

print('\n📢 Para PROMOCIONES MASIVAS (mayor soporte):')
for _, r in top3_sup.iterrows():
    print(f"   {r['antecedents_str']} → {r['consequents_str']}  (sup={r['support']:.3f})")

---
## 10. Ejercicios

### 🎚️ Ejercicio 1 — Sensibilidad al umbral
Cambia `MIN_SUPPORT` a 0.03, 0.08 y 0.15. Para cada valor, anota:
- ¿Cuántas reglas se generan?
- ¿Qué lift promedio tienen?
- ¿Aparecen reglas espurias (lift cercano a 1)?

### 🔍 Ejercicio 2 — Reglas dirigidas
Cambia `TARGET_ITEM` a `'cerveza'` y `'chocolate'`. ¿Qué diferencias encuentras en los patrones de compra?

### 📊 Ejercicio 3 — Confianza vs Lift
Encuentra una regla donde `confidence` sea alta (>0.6) pero `lift` sea cercano a 1. ¿Por qué no es útil esa regla aunque tenga alta confianza?

### ⚖️ Ejercicio 4 — Asimetría
Para la regla `{pañales} → {cerveza}`, ¿cuál es la confianza de la regla inversa `{cerveza} → {pañales}`? ¿Cuál dirección es más útil para marketing?

### 🧮 Ejercicio 5 — Tu propio dataset
Modifica el diccionario `ITEM_PROBS` para simular un **comercio electrónico de tecnología** (laptops, mouses, teclados, audífonos, etc.) y vuelve a ejecutar el análisis completo.